In [26]:
import os
import operator
from typing import TypedDict, Annotated, List

# --- Configuración de API ---
# ¡Asegúrate de poner tu clave real de Tavily aquí!
os.environ["TAVILY_API_KEY"] = "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" 

# --- Importaciones de LangChain y LangGraph ---
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END

In [27]:
# --- 1. Definir el Estado (Memoria) ---
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    notes: Annotated[list[str], operator.add]

# --- 2. Definir las Herramientas ---
search_tool = TavilySearchResults(max_results=2)

@tool
def save_note(content: str):
    """
    Guarda una nota de texto en la memoria contextual del agente 
    para referencia futura.
    """
    print(f"--- Herramienta save_note invocada con: '{content}' ---")
    return f"Nota guardada exitosamente: '{content}'"

tools = [search_tool, save_note]
tools_by_name = {tool.name: tool for tool in tools}

In [28]:
tu_endpoint_de_la_upv = "https://api.poligpt.upv.es"  # Pega aquí la URL del endpoint
tu_api_key_de_la_upv = "[CLAVE_ELIMINADA]"    # Pega aquí tu API key

# Configuramos el LLM de LangChain
llm = ChatOpenAI(
    # 1. El nombre del modelo que te dio la UPV
    model="llama", 
    
    # 2. Tu clave de API
    api_key=tu_api_key_de_la_upv,
    
    # 3. La URL base del servidor de PoliGPT
    base_url=tu_endpoint_de_la_upv 
)

# --- ¡El resto de tu código no cambia! ---
# LangGraph funcionará perfectamente con esta configuración.
llm_with_tools = llm.bind_tools(tools)

In [29]:
# --- 4. Definir los Nodos del Grafo ---

def agent_node(state: AgentState):
    print("--- Llamando al Agente (Mistral) ---")
    response = llm_with_tools.invoke(state['messages'])
    return {"messages": [response]}


# --- REEMPLAZA TU action_node CON ESTE ---

def action_node(state: AgentState):
    print("--- Llamando al Nodo de Acción (Herramientas) ---")
    last_message = state['messages'][-1]
    tool_calls = last_message.tool_calls
    
    # Listas para guardar las salidas
    tool_messages = []
    notes_to_add = []

    # Iteramos sobre cada llamada a herramienta que decidió el LLM
    for call in tool_calls:
        print(f"Ejecutando herramienta: {call['name']}")
        
        # Encontrar la herramienta correcta en nuestro diccionario
        tool_to_call = tools_by_name.get(call['name'])
        
        if not tool_to_call:
            # Caso de error: el LLM inventó una herramienta
            output = f"Error: Herramienta '{call['name']}' no encontrada."
        else:
            try:
                # Ejecutar la herramienta con sus argumentos
                output = tool_to_call.invoke(call['args'])
            except Exception as e:
                output = f"Error al ejecutar la herramienta: {e}"

        # --- Lógica de Memoria Contextual ---
        # Si la herramienta fue 'save_note', guardamos la nota
        if call['name'] == 'save_note':
            # Guardamos el *argumento* de entrada, no la salida
            notes_to_add.append(call['args']['content'])
        # -------------------------------------

        # Añadimos la salida de la herramienta a la lista de mensajes
        tool_messages.append(
            ToolMessage(content=str(output), tool_call_id=call['id'])
        )
    
    # Devolvemos las actualizaciones al estado
    return {
        "messages": tool_messages,
        "notes": notes_to_add
    }

In [30]:
# --- 5. Construir y Compilar el Grafo ---

def should_continue(state: AgentState):
    last_message = state['messages'][-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("action", action_node)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"continue": "action", "end": END}
)
workflow.add_edge("action", "agent")

app = workflow.compile()

In [31]:
# --- 6. Ejecutar el Agente ---

config = {"recursion_limit": 10}
inputs = [
    HumanMessage(content="Hola, ¿quién ganó la final de la Champions League de 2024?"),
    HumanMessage(content="¿Cuántas champions tiene ese equipo?"),
    HumanMessage(content="¿Cuando se descubrió América?"),
]

print("--- Ejecutando el Agente con Mistral ---")

# Este estado se mantiene vivo fuera del grafo
current_state = {"messages": [], "notes": []}

for i, msg in enumerate(inputs):
    print(f"\n--- TURNO DE CONVERSACIÓN {i+1} ---")
    print(f"Usuario: {msg.content}")
    
    # Añadimos el nuevo mensaje del usuario al estado
    current_state["messages"].append(msg)
    
    final_response = None
    # Hacemos streaming de la ejecución del grafo
    for step in app.stream(current_state, config=config):
        node_name = list(step.keys())[0]
        node_output = step[node_name]
        
        print(f"Nodo: {node_name}")
        
        # Actualizamos nuestro estado externo con lo que devuelve el nodo
        current_state["messages"].extend(node_output.get("messages", []))
        current_state["notes"].extend(node_output.get("notes", []))
        
        # Guardamos la respuesta final del agente
        if node_name == "agent" and not node_output["messages"][-1].tool_calls:
            final_response = node_output["messages"][-1]

    print(f"Agente: {final_response.content}")
    print("--- Estado Actual de la Memoria (Notas) ---")
    print(current_state["notes"])

--- Ejecutando el Agente con Mistral ---

--- TURNO DE CONVERSACIÓN 1 ---
Usuario: Hola, ¿quién ganó la final de la Champions League de 2024?
--- Llamando al Agente (Mistral) ---
Nodo: agent
--- Llamando al Nodo de Acción (Herramientas) ---
Ejecutando herramienta: tavily_search_results_json
Nodo: action
--- Llamando al Agente (Mistral) ---
Nodo: agent
Agente: El ganador de la final de la Champions League de 2024 fue el Real Madrid, que derrotó al Borussia Dortmund con un marcador de 2-0 en el Wembley Stadium de Londres, Inglaterra, el 1 de junio de 2024.
--- Estado Actual de la Memoria (Notas) ---
[]

--- TURNO DE CONVERSACIÓN 2 ---
Usuario: ¿Cuántas champions tiene ese equipo?
--- Llamando al Agente (Mistral) ---
Nodo: agent
--- Llamando al Nodo de Acción (Herramientas) ---
Ejecutando herramienta: tavily_search_results_json
Nodo: action
--- Llamando al Agente (Mistral) ---
Nodo: agent
Agente: El Real Madrid ha ganado la Champions League un total de 15 veces, lo que lo convierte en el 